# The Price is Right

Today we build a more complex solution for estimating prices of goods.

1. This notebook: create a RAG database with our 400,000 training data
2. Day 2.1 notebook: visualize in 2D
3. Day 2.2 notebook: visualize in 3D
4. Day 2.3 notebook: build and test a RAG pipeline with GPT-4o-mini
5. Day 2.4 notebook: (a) bring back our Random Forest pricer (b) Create a Ensemble pricer that allows contributions from all the pricers

Phew! That's a lot to get through in one day!

## PLEASE NOTE:

We already have a very powerful product estimator with our proprietary, fine-tuned LLM. Most people would be very satisfied with that! The main reason we're adding these extra steps is to deepen your expertise with RAG and with Agentic workflows.


In [4]:
# imports

import os
import re
import math
import json
from tqdm import tqdm
import random
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [5]:
# environment

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')
DB = "products_vectorstore"

In [3]:
# Log in to HuggingFace

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# Another import after Logging in to Hugging Face - thank you Trung N.!

from items import Item

## Back to the pkl files

Much as we enjoyed data curating in week 6, we probably don't want to go through that whole process again!

Let's reuse the pkl files we created then. Either copy the files `train.pkl` and `test.pkl` from the Week 6 folder into this Week 8 folder, or you can also download them from here:

https://drive.google.com/drive/folders/1f_IZGybvs9o0J5sb3xmtTEQB3BXllzrW?usp=drive_link

In [5]:
# With train.pkl in this folder, you can run this:

with open('data/train.pkl', 'rb') as file:
    train = pickle.load(file)

In [6]:
train[0].prompt

'How much does this cost to the nearest dollar?\n\nDelphi FG0166 Fuel Pump Module\nDelphi brings 80 years of OE Heritage into each Delphi pump, ensuring quality and fitment for each Delphi part. Part is validated, tested and matched to the right vehicle application Delphi brings 80 years of OE Heritage into each Delphi assembly, ensuring quality and fitment for each Delphi part Always be sure to check and clean fuel tank to avoid unnecessary returns Rigorous OE-testing ensures the pump can withstand extreme temperatures Brand Delphi, Fit Type Vehicle Specific Fit, Dimensions LxWxH 19.7 x 7.7 x 5.1 inches, Weight 2.2 Pounds, Auto Part Position Unknown, Operation Mode Mechanical, Manufacturer Delphi, Model FUEL PUMP, Dimensions 19.7\n\nPrice is $227.00'

# Now create a Chroma Datastore

In Week 5, we created a Chroma datastore with 123 documents representing chunks of objects from our fictional company Insurellm.

Now we will create a Chroma datastore with 400,000 products from our training dataset! It's getting real!

Note that we won't be using LangChain, but the API is very straightforward and consistent with before.

Special note: if Chroma crashes and you're a Windows user, you should try rolling back to an earlier version of the Chroma library with:  
`!pip install chromadb==0.5.0`  
With many thanks to student Kelly Z. for finding this out and pointing to the GitHub issue [here](https://github.com/chroma-core/chroma/issues/2513). 
But try first without reverting Chroma.

In [6]:
client = chromadb.PersistentClient(path=DB)

In [8]:
# Check if the collection exists and delete it if it does
collection_name = "products"

# For old versions of Chroma, use this line instead of the subsequent one
# existing_collection_names = [collection.name for collection in client.list_collections()]
existing_collection_names = client.list_collections()

if collection_name in existing_collection_names:
    client.delete_collection(collection_name)
    print(f"Deleted existing collection: {collection_name}")

collection = client.create_collection(collection_name)

# Introducing the SentenceTransfomer

The all-MiniLM is a very useful model from HuggingFace that maps sentences & paragraphs to a 384 dimensional dense vector space and is ideal for tasks like semantic search.

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

It can run pretty quickly locally.

Last time we used OpenAI embeddings to produce vector embeddings. Benefits compared to OpenAI embeddings:
1. It's free and fast!
3. We can run it locally, so the data never leaves our box - might be useful if you're building a personal RAG


In [9]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [10]:
# Pass in a list of texts, get back a numpy array of vectors

vector = model.encode(["Well hi there"])[0]
vector

array([-9.46715772e-02,  4.27620076e-02,  5.51620498e-02, -5.10970887e-04,
        1.16202980e-02, -6.80130422e-02,  2.76405867e-02,  6.06974587e-02,
        2.88531017e-02, -1.74128339e-02, -4.94346246e-02,  2.30993051e-02,
       -1.28614437e-02, -4.31402586e-02,  2.17509698e-02,  4.26548198e-02,
        5.10500371e-02, -7.79727101e-02, -1.23247243e-01,  3.67455892e-02,
        4.54119081e-03,  9.47938412e-02, -5.53098843e-02,  1.70641653e-02,
       -2.92872209e-02, -4.47124578e-02,  2.06784271e-02,  6.39320314e-02,
        2.27427725e-02,  4.87789586e-02, -2.33500893e-03,  4.72859032e-02,
       -2.86259297e-02,  2.30624489e-02,  2.45130286e-02,  3.95681411e-02,
       -4.33176868e-02, -1.02316663e-01,  2.79874611e-03,  2.39304528e-02,
        1.61556639e-02, -8.99080746e-03,  2.07256041e-02,  6.40123039e-02,
        6.89179078e-02, -6.98361844e-02,  2.89758621e-03, -8.10989439e-02,
        1.71122830e-02,  2.50659091e-03, -1.06529087e-01, -4.87733483e-02,
       -1.67762171e-02, -

In [11]:
# Quick sidebar - extra to the videos - a function to compare vectors

import numpy as np
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def how_similar(text1, text2):
    vector1, vector2 = model.encode([text1, text2])
    similarity = cosine_similarity(vector1, vector2)
    print(f"Similarity between {text1} and {text2} is {similarity*100:.1f}%")

In [16]:
# And let's see how adding a few words to the context can change things up!

how_similar("Java", "Python")
how_similar("Java", "mug")
how_similar("Cup of Java", "mug")

Similarity between Java and Python is 45.0%
Similarity between Java and mug is 25.8%
Similarity between Cup of Java and mug is 49.3%


In [13]:
# OK back to the main story - let's make something we can vectorize

def description(item):
    text = item.prompt.replace("How much does this cost to the nearest dollar?\n\n", "")
    return text.split("\n\nPrice is $")[0]

In [14]:
description(train[0])

'Delphi FG0166 Fuel Pump Module\nDelphi brings 80 years of OE Heritage into each Delphi pump, ensuring quality and fitment for each Delphi part. Part is validated, tested and matched to the right vehicle application Delphi brings 80 years of OE Heritage into each Delphi assembly, ensuring quality and fitment for each Delphi part Always be sure to check and clean fuel tank to avoid unnecessary returns Rigorous OE-testing ensures the pump can withstand extreme temperatures Brand Delphi, Fit Type Vehicle Specific Fit, Dimensions LxWxH 19.7 x 7.7 x 5.1 inches, Weight 2.2 Pounds, Auto Part Position Unknown, Operation Mode Mechanical, Manufacturer Delphi, Model FUEL PUMP, Dimensions 19.7'

## Now we populate our RAG datastore

The next cell populates the 400,000 items in Chroma.

Feel free to reduce the number of documents if this takes too long! You can change to:  
`NUMBER_OF_DOCUMENTS = 20000`  
And that's plenty for a perfectly good RAG pipeline.

Just note that if you interrupt the below cell while it's running, you might need to clear out the Chroma datastore (by rerunning the earlier cell that deletes the collection), before you run it again. Otherwise it will complain that there are existing documents with the same ID.

In [15]:
NUMBER_OF_DOCUMENTS = len(train)

# Uncomment if you'd rather not wait for the full 400,000
# NUMBER_OF_DOCUMENTS = 20000

for i in tqdm(range(0, NUMBER_OF_DOCUMENTS, 1000)):
    documents = [description(item) for item in train[i: i+1000]]
    vectors = model.encode(documents).astype(float).tolist()
    metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
    ids = [f"doc_{j}" for j in range(i, i+len(documents))]
    metadatas = [{"category": str(item.category), "price": float(item.price)} for item in train[i: i+1000]]
    collection.add(
        ids=ids,
        documents=documents,
        embeddings=vectors,
        metadatas=metadatas
    )

100%|██████████| 400/400 [3:16:15<00:00, 29.44s/it]  


In [6]:
import chromadb

DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)

# Check if the collection exists and delete it if it does
collection_name = "products"

# For old versions of Chroma, use this line instead of the subsequent one
# existing_collection_names = [collection.name for collection in client.list_collections()]
existing_collections = client.list_collections()
print(existing_collections[0].name)
if collection_name in [col.name for col in existing_collections]:
    print(f"Collection \"{collection_name}\" exists.")

print(existing_collections[0].peek(limit=1))
    

products
Collection "products" exists.
{'ids': ['doc_0'], 'embeddings': array([[-5.68037406e-02,  3.79323848e-02,  4.97085825e-02,
        -5.26031554e-02,  3.39973681e-02, -1.41842635e-02,
         4.60331552e-02,  9.69829187e-02, -2.92877350e-02,
        -1.26629338e-01, -4.81041446e-02, -1.42702106e-02,
         3.09890206e-03, -3.90520617e-02,  8.63330625e-03,
         1.21714463e-02, -4.41983826e-02, -7.34628960e-02,
        -2.35757176e-02, -4.19214787e-03,  2.91597042e-02,
        -1.01110553e-02,  3.21704857e-02,  2.71455999e-02,
        -6.03994168e-02, -1.90803483e-02,  3.23044509e-02,
         1.05472356e-01, -3.87021862e-02, -6.45612255e-02,
         3.19408029e-02,  2.99944961e-03, -1.61847798e-03,
         4.52610627e-02,  3.58513817e-02,  4.18819971e-02,
        -5.32129072e-02, -2.19634678e-02, -4.30964790e-02,
        -8.05794299e-02,  5.46365138e-03, -3.49104665e-02,
         3.61400358e-02,  7.01950714e-02,  3.01539227e-02,
         1.16034886e-02, -5.55701107e-02,  